# Gender Classification Using an Optimized CNN

This notebook implements a VGG-inspired convolutional neural network for binary gender classification from facial images. The architecture uses convolutional blocks, batch normalization, global average pooling, and dropout to reduce the number of trainable parameters and help control overfitting.

## 1. Import Libraries

Import the libraries required for image processing, data preparation, model construction, training, and evaluation.

In [ ]:
import os
import pickle
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 2. Connect Google Drive and Define Dataset Paths

Mount Google Drive and specify the directory containing the facial images. The dataset is expected to contain filenames in which the second underscore-separated field represents the binary class label.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

DATASET_PATH = '/content/gdrive/My Drive/photo'
SPLIT_PATH = '/content/gdrive/My Drive/dataset_split'
MODEL_PATH = '/content/gdrive/My Drive/models'

os.makedirs(SPLIT_PATH, exist_ok=True)
os.makedirs(MODEL_PATH, exist_ok=True)

if not os.path.isdir(DATASET_PATH):
    raise FileNotFoundError(f'Dataset directory not found: {DATASET_PATH}')

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
files = sorted([
    filename for filename in os.listdir(DATASET_PATH)
    if filename.lower().endswith(image_extensions)
])

MAX_IMAGES = 1200

if len(files) < MAX_IMAGES:
    raise ValueError(
        f'At least {MAX_IMAGES} images are required. Found: {len(files)}'
    )

files = files[:MAX_IMAGES]
print('Number of selected images:', len(files))

## 3. Extract Image Labels

Extract the binary class label from the second field of each filename. Labels must be encoded as 0 or 1.

In [ ]:
genders = []

for filename in files:
    fields = filename.split('_')

    if len(fields) < 2:
        raise ValueError(f'Unexpected filename format: {filename}')

    try:
        gender = int(fields[1])
    except ValueError:
        raise ValueError(f'Invalid label in filename: {filename}')

    if gender not in (0, 1):
        raise ValueError(f'Expected label 0 or 1, found {gender}: {filename}')

    genders.append(gender)

target = np.array(genders, dtype=np.int32)

print('Total images:', len(target))
print('Number of class 0:', np.sum(target == 0))
print('Number of class 1:', np.sum(target == 1))

## 4. Split the Dataset

Split the dataset into training and testing subsets using stratified sampling. This ensures that the class proportions are approximately preserved in both subsets. The indices are saved to Google Drive for reproducibility.

In [ ]:
all_indices = np.arange(len(files))

train_ind, test_ind = train_test_split(
    all_indices,
    test_size=0.1,
    random_state=SEED,
    stratify=target
)

with open(os.path.join(SPLIT_PATH, 'train_indfile'), 'wb') as f:
    pickle.dump(train_ind, f)

with open(os.path.join(SPLIT_PATH, 'test_indfile'), 'wb') as f:
    pickle.dump(test_ind, f)

y_train_all = target[train_ind]
y_test = target[test_ind]

print('Training samples:', len(train_ind))
print('Testing samples:', len(test_ind))
print('Class distribution in training:', np.bincount(y_train_all, minlength=2))
print('Class distribution in testing:', np.bincount(y_test, minlength=2))

## 5. Load and Preprocess Images

Load facial images using OpenCV, convert them from BGR to RGB, resize them to 224 × 224 pixels, and normalize their pixel values to the range [0, 1].

In [ ]:
IMAGE_SIZE = 224

def load_images(indices):
    images = []

    for idx in indices:
        image_path = os.path.join(DATASET_PATH, files[idx])
        image = cv2.imread(image_path, cv2.IMREAD_COLOR)

        if image is None:
            raise ValueError(f'Could not read image: {image_path}')

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        images.append(image)

    return np.asarray(images, dtype=np.float32) / 255.0

x_train_all = load_images(train_ind)
x_test = load_images(test_ind)

print('Training images shape:', x_train_all.shape)
print('Testing images shape:', x_test.shape)

plt.figure(figsize=(5, 5))
plt.imshow(x_train_all[0])
plt.title(f'Class label: {y_train_all[0]}')
plt.axis('off')
plt.show()

## 6. Create Training and Validation Sets

Separate a validation subset from the training data. The test set remains untouched during model training and model selection.

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train_all,
    y_train_all,
    test_size=0.1,
    random_state=SEED,
    stratify=y_train_all
)

print('Training set:', x_train.shape, y_train.shape)
print('Validation set:', x_val.shape, y_val.shape)
print('Testing set:', x_test.shape, y_test.shape)

## 7. Define Data Augmentation

Apply light data augmentation during training to improve generalization. The transformations are applied only to training images and do not modify the validation or test sets.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1)
], name='data_augmentation')

## 8. Build the Optimized CNN Architecture

Construct a VGG-inspired CNN using convolutional blocks with increasing filter counts. Batch normalization improves optimization, while global average pooling reduces the number of parameters compared with flattening the complete feature maps. Dropout is used to help reduce overfitting.

In [ ]:
def build_cnn_model(input_shape=(224, 224, 3)):
    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    # Block 1
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 4
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(256, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(2, activation='softmax')(x)

    return models.Model(inputs=inputs, outputs=outputs, name='Optimized_CNN')

model = build_cnn_model()
model.summary()

## 9. Compile the Model

Compile the CNN using the Adam optimizer and sparse categorical cross-entropy. Accuracy is monitored during training and validation.

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 10. Define Training Callbacks

Use early stopping to prevent unnecessary training, reduce the learning rate when validation performance stops improving, and save the model with the best validation loss.

In [ ]:
MODEL_FILE = os.path.join(MODEL_PATH, 'optimized_gender_cnn.keras')

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6
    ),
    ModelCheckpoint(
        MODEL_FILE,
        monitor='val_loss',
        save_best_only=True
    )
]

print('Best model will be saved to:', MODEL_FILE)

## 11. Train the CNN

Train the optimized CNN using the training set and monitor its performance on the validation set. Early stopping may terminate training before the maximum number of epochs.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    batch_size=32,
    epochs=100,
    callbacks=callbacks,
    shuffle=True
)

## 12. Visualize Training Performance

Plot training and validation accuracy and loss across epochs to inspect model convergence and possible overfitting.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 13. Load the Best Saved Model

Load the model checkpoint selected using validation loss. This ensures that evaluation uses the best saved model rather than necessarily the weights from the final training epoch.

In [ ]:
if not os.path.isfile(MODEL_FILE):
    raise FileNotFoundError('The saved model was not found. Run training first.')

best_model = tf.keras.models.load_model(MODEL_FILE)
best_model.summary()

## 14. Evaluate the Model on Training and Test Sets

Evaluate the selected model on the training and held-out test sets. The test set is used only for final performance evaluation.

In [ ]:
train_loss, train_accuracy = best_model.evaluate(
    x_train, y_train, batch_size=32, verbose=1
)

test_loss, test_accuracy = best_model.evaluate(
    x_test, y_test, batch_size=32, verbose=1
)

print(f'Training accuracy: {train_accuracy * 100:.2f}%')
print(f'Test accuracy: {test_accuracy * 100:.2f}%')
print(f'Test loss: {test_loss:.4f}')

## 15. Generate Predictions and Calculate Classification Metrics

Generate class probabilities for the test images and convert them into predicted class labels using argmax. Calculate precision, recall, F1-score, and accuracy.

In [ ]:
predicted_probabilities = best_model.predict(x_test, batch_size=32, verbose=1)
predicted_labels = np.argmax(predicted_probabilities, axis=1)

precision = precision_score(y_test, predicted_labels, zero_division=0)
recall = recall_score(y_test, predicted_labels, zero_division=0)
f1 = f1_score(y_test, predicted_labels, zero_division=0)
accuracy = accuracy_score(y_test, predicted_labels)

print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print('Accuracy:', accuracy)

print('\nClassification Report:')
print(classification_report(y_test, predicted_labels, zero_division=0))

## 16. Display the Confusion Matrix

Visualize the confusion matrix to inspect the number of correctly and incorrectly classified samples in each class.

In [ ]:
cm = confusion_matrix(y_test, predicted_labels)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Class 0', 'Class 1']
)

disp.plot(cmap='Blues')
plt.title('Confusion Matrix - Optimized CNN')
plt.show()

## 17. Count Samples in Each Class

Display the number of samples belonging to each class in the training and testing sets.

In [ ]:
print('Number of class 0 in training:', np.sum(y_train_all == 0))
print('Number of class 1 in training:', np.sum(y_train_all == 1))

print('Number of class 0 in testing:', np.sum(y_test == 0))
print('Number of class 1 in testing:', np.sum(y_test == 1))

## 18. Save the Final Evaluation Results

Store the final classification metrics in a structured table for reporting and later comparison.

In [ ]:
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score', 'Test Loss'],
    'Value': [accuracy, precision, recall, f1, test_loss]
})

display(results)

RESULTS_FILE = os.path.join(MODEL_PATH, 'gender_classification_results.csv')
results.to_csv(RESULTS_FILE, index=False)

print('Results saved to:', RESULTS_FILE)

## Conclusion

An optimized VGG-inspired CNN was developed for binary gender classification from facial images. The model uses convolutional feature extraction, batch normalization, global average pooling, and dropout. Data augmentation and early stopping are used to improve generalization and reduce overfitting. Final performance is reported using accuracy, precision, recall, F1-score, and a confusion matrix.